# Random Forest Power Demand Forecasting

## 1. Model Development

This section develops three independent Random Forest Regression models to forecast electricity consumption for Zone 1, Zone 2 and Zone 3.

The forecasting horizon is one hour ahead.

The dataset is divided chronologically into training (70%), validation (15%) and testing (15%) sets to prevent future information from entering the training process.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load engineered dataset
df = pd.read_csv("../outputs/engineered_power_data.csv")
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Identify target columns
target_cols = [col for col in df.columns if col.endswith("_Target")]

# Identify input features
feature_cols = [
    col for col in df.columns
    if col not in target_cols and col != 'DateTime'
]

print("Dataset loaded:", df.shape)
print("\nTarget columns:", target_cols)
print("\nNumber of input features:", len(feature_cols))
print("\nTarget verification:")
display(df[target_cols].head())

Dataset loaded: (52266, 45)

Target columns: ['Zone 1_Target', 'Zone 2 _Target', 'Zone 3 _Target']

Number of input features: 41

Target verification:


,Zone 1_Target,Zone 2 _Target,Zone 3 _Target
0,23629.36709,15720.36474,15643.37349
1,23118.98734,15304.55927,15307.95181
2,22736.20253,15136.77812,15093.97590
3,22049.62025,14637.08207,14972.53012
4,21855.18987,14414.58967,14833.73494


In [2]:
# Leakage-safe chronological split

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Remove six rows at each boundary
train_df = df.iloc[:train_end - 6].copy()
val_df = df.iloc[train_end:val_end - 6].copy()
test_df = df.iloc[val_end:].copy()

print("TRAINING RECORDS:", len(train_df))
print("VALIDATION RECORDS:", len(val_df))
print("TESTING RECORDS:", len(test_df))

print("\nTraining period:")
print(train_df['DateTime'].min(), "to", train_df['DateTime'].max())

print("\nValidation period:")
print(val_df['DateTime'].min(), "to", val_df['DateTime'].max())

print("\nTesting period:")
print(test_df['DateTime'].min(), "to", test_df['DateTime'].max())

TRAINING RECORDS: 36580
VALIDATION RECORDS: 7834
TESTING RECORDS: 7840

Training period:
2017-01-02 00:00:00 to 2017-09-13 00:30:00

Validation period:
2017-09-13 01:40:00 to 2017-11-06 11:10:00

Testing period:
2017-11-06 12:20:00 to 2017-12-30 22:50:00


In [4]:

# ==========================================
# RANDOM FOREST MODEL TRAINING
# ==========================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

os.makedirs("../models", exist_ok=True)
os.makedirs("../outputs", exist_ok=True)

# Recommended hyperparameter search space
param_grid = {
    "n_estimators": [200, 400, 800],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5, 1.0],
    "bootstrap": [True]
}

results = []
trained_models = {}

# Time-series cross-validation
tscv = TimeSeriesSplit(n_splits=3, gap=6)

for target in target_cols:

    print("\n" + "=" * 60)
    print("TRAINING MODEL:", target)
    print("=" * 60)

    X_train = train_df[feature_cols]
    y_train = train_df[target]

    X_val = val_df[feature_cols]
    y_val = val_df[target]

    rf = RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    )

    search = RandomizedSearchCV(
        estimator=rf,
        param_distributions=param_grid,
        n_iter=8,
        cv=tscv,
        scoring="neg_mean_absolute_error",
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    search.fit(X_train, y_train)

    best_model = search.best_estimator_

    # Validation evaluation only
    val_pred = best_model.predict(X_val)

    val_mae = mean_absolute_error(y_val, val_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    val_r2 = r2_score(y_val, val_pred)

    model_name = target.replace(" ", "_")
    model_path = f"../models/{model_name}_rf.joblib"

    joblib.dump(best_model, model_path)

    trained_models[target] = best_model

    results.append({
        "Zone": target,
        "Best Parameters": str(search.best_params_),
        "Best CV MAE": -search.best_score_,
        "Validation MAE": val_mae,
        "Validation RMSE": val_rmse,
        "Validation R2": val_r2
    })

    print("\nBest Parameters:", search.best_params_)
    print("Validation MAE:", round(val_mae, 3))
    print("Validation RMSE:", round(val_rmse, 3))
    print("Validation R2:", round(val_r2, 4))
    print("Model saved:", model_path)

# Save training and tuning results
results_df = pd.DataFrame(results)

results_df.to_csv(
    "../outputs/training_tuning_log.csv",
    index=False
)

print("\nALL THREE MODELS TRAINED SUCCESSFULLY!")
print("Models saved in models folder.")
print("Tuning results saved in outputs folder.")
print("\nTEST DATA HAS NOT BEEN USED.")
display(results_df.round(4))



TRAINING MODEL: Zone 1_Target
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best Parameters: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True}
Validation MAE: 1048.654
Validation RMSE: 1544.656
Validation R2: 0.9452
Model saved: ../models/Zone_1_Target_rf.joblib

TRAINING MODEL: Zone 2 _Target
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best Parameters: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 0.5, 'max_depth': 10, 'bootstrap': True}
Validation MAE: 719.543
Validation RMSE: 1131.177
Validation R2: 0.9401
Model saved: ../models/Zone_2__Target_rf.joblib

TRAINING MODEL: Zone 3 _Target
Fitting 3 folds for each of 8 candidates, totalling 24 fits

Best Parameters: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True}
Validation MAE: 990.683
Validation RMSE: 1316.523
V

,Zone,Best Parameters,Best CV MAE,Validation MAE,Validation RMSE,Validation R2
0,Zone 1_Target,"{'n_estimators': 400, 'min_samples_split': 10,...",1367.1949,1048.6540,1544.6565,0.9452
1,Zone 2 _Target,"{'n_estimators': 200, 'min_samples_split': 2, ...",1028.7413,719.5434,1131.1773,0.9401
2,Zone 3 _Target,"{'n_estimators': 400, 'min_samples_split': 10,...",1221.1907,990.6834,1316.5227,0.8295
